# 🚨 RescueVision — Prioritizing Disaster-Affected Buildings
### CNN + Transfer Learning for Building Damage Classification

**Objective:** Classify satellite images of buildings as *damaged* or *not damaged* after a hurricane, and assign rescue priority levels.

**Approach:**
- Transfer Learning using **ResNet50** (pre-trained on ImageNet)
- Binary classification: `damage` vs `no_damage`
- Priority mapping for rescue operations

**Dataset:** Satellite Images of Hurricane Damage (Hurricane Harvey, Texas)

---
## 1. Import Libraries

In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 2. Dataset Preparation

The downloaded dataset has `test/` (balanced) and `test_another/` (unbalanced) folders.  
We'll combine them and create proper **train / validation / test** splits (70/15/15).

In [ ]:
# --- Configuration ---
BASE_DIR = r'archive(1)'  # Change this if your dataset is elsewhere
SPLIT_DIR = 'dataset_split'  # Where we'll create train/val/test splits
IMG_SIZE = (224, 224)  # ResNet50 input size
BATCH_SIZE = 32
CLASSES = ['damage', 'no_damage']

print(f"Base directory: {os.path.abspath(BASE_DIR)}")
print(f"Split directory: {os.path.abspath(SPLIT_DIR)}")

In [ ]:
# --- Count original images ---
print("Original dataset distribution:")
print("=" * 40)

for folder in ['test', 'test_another']:
    for cls in CLASSES:
        path = os.path.join(BASE_DIR, folder, cls)
        if os.path.exists(path):
            count = len(os.listdir(path))
            print(f"  {folder}/{cls}: {count} images")

# Collect all image paths per class
all_images = {'damage': [], 'no_damage': []}

for folder in ['test', 'test_another']:
    for cls in CLASSES:
        path = os.path.join(BASE_DIR, folder, cls)
        if os.path.exists(path):
            for img_name in os.listdir(path):
                img_path = os.path.join(path, img_name)
                if os.path.isfile(img_path):
                    all_images[cls].append(img_path)

print(f"\nTotal damage images: {len(all_images['damage'])}")
print(f"Total no_damage images: {len(all_images['no_damage'])}")
print(f"Total images: {len(all_images['damage']) + len(all_images['no_damage'])}")

In [ ]:
# --- Create balanced train/val/test splits ---
# Balance classes by undersampling the majority class
min_count = min(len(all_images['damage']), len(all_images['no_damage']))
print(f"Balancing classes to {min_count} images each")

for cls in CLASSES:
    random.shuffle(all_images[cls])
    all_images[cls] = all_images[cls][:min_count]

# Split ratios
train_ratio, val_ratio, test_ratio = 0.70, 0.15, 0.15

splits = {}
for cls in CLASSES:
    n = len(all_images[cls])
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)
    
    splits[cls] = {
        'train': all_images[cls][:train_end],
        'val': all_images[cls][train_end:val_end],
        'test': all_images[cls][val_end:]
    }

# Copy files into split directories
if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

for split_name in ['train', 'val', 'test']:
    for cls in CLASSES:
        dest_dir = os.path.join(SPLIT_DIR, split_name, cls)
        os.makedirs(dest_dir, exist_ok=True)
        for src_path in splits[cls][split_name]:
            filename = os.path.basename(src_path)
            dest_path = os.path.join(dest_dir, filename)
            # Use copy to avoid modifying original data
            shutil.copy2(src_path, dest_path)

# Print split summary
print("\nDataset split summary:")
print("=" * 40)
for split_name in ['train', 'val', 'test']:
    for cls in CLASSES:
        path = os.path.join(SPLIT_DIR, split_name, cls)
        count = len(os.listdir(path))
        print(f"  {split_name}/{cls}: {count} images")

---
## 3. Data Loading & Augmentation

In [ ]:
# --- Data Generators ---
# Training data with augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    shear_range=0.1,
    fill_mode='nearest'
)

# Validation & Test data — only preprocessing, no augmentation
val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"\nClass indices: {train_generator.class_indices}")

In [ ]:
# --- Visualize sample images ---
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images from Training Set', fontsize=16, fontweight='bold')

class_names = list(train_generator.class_indices.keys())

for row, cls in enumerate(class_names):
    cls_dir = os.path.join(SPLIT_DIR, 'train', cls)
    sample_images = random.sample(os.listdir(cls_dir), 5)
    for col, img_name in enumerate(sample_images):
        img = Image.open(os.path.join(cls_dir, img_name))
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls, fontsize=10)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sample_images.png")

---
## 4. Build the Model — ResNet50 + Transfer Learning

In [ ]:
# --- Load Pre-trained ResNet50 (without top classification layer) ---
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze all layers in the base model initially
base_model.trainable = False

print(f"Base model: ResNet50")
print(f"Total layers: {len(base_model.layers)}")
print(f"Trainable parameters (frozen): {base_model.count_params():,}")

In [ ]:
# --- Build the complete model ---
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

---
## 5. Train the Model

**Phase 1:** Train only the custom classification head (base model frozen)  
**Phase 2:** Fine-tune the last few layers of ResNet50

In [ ]:
# --- Phase 1: Train the classification head ---
print("=" * 50)
print("PHASE 1: Training classification head (base frozen)")
print("=" * 50)

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

history_phase1 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# --- Phase 2: Fine-tune the last few layers of ResNet50 ---
print("\n" + "=" * 50)
print("PHASE 2: Fine-tuning last 20 layers of ResNet50")
print("=" * 50)

# Unfreeze the last 20 layers of the base model
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

trainable_count = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f"Trainable parameters after unfreezing: {trainable_count:,}")

# Recompile with a lower learning rate for fine-tuning
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_phase2 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

---
## 6. Training History Visualization

In [ ]:
# --- Combine training history from both phases ---
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined

full_history = combine_histories(history_phase1, history_phase2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(full_history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(full_history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].axvline(x=len(history_phase1.history['accuracy'])-0.5, color='red', 
                linestyle='--', alpha=0.7, label='Fine-tuning starts')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(full_history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(full_history['val_loss'], label='Val Loss', linewidth=2)
axes[1].axvline(x=len(history_phase1.history['loss'])-0.5, color='red', 
                linestyle='--', alpha=0.7, label='Fine-tuning starts')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_history.png")

---
## 7. Model Evaluation

In [ ]:
# --- Evaluate on test set ---
print("Evaluating on test set...")
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f"\n{'='*40}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"{'='*40}")

In [ ]:
# --- Predictions & Classification Report ---
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()
y_true = test_generator.classes

# Class names from generator
class_labels = list(test_generator.class_indices.keys())

print("\nClassification Report:")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=class_labels))

In [ ]:
# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_labels, yticklabels=class_labels,
            annot_kws={'size': 16})
ax.set_title('Confusion Matrix', fontsize=16, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
# --- ROC Curve ---
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', linewidth=1, linestyle='--', label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roc_curve.png")

---
## 8. Rescue Priority Mapping

Map model predictions to actionable **rescue priority levels**:

| Confidence (Damage) | Priority Level | Action |
|---|---|---|
| > 90% | 🔴 **CRITICAL (P1)** | Immediate rescue deployment |
| 70% – 90% | 🟠 **HIGH (P2)** | Urgent assessment needed |
| 50% – 70% | 🟡 **MEDIUM (P3)** | Schedule inspection |
| < 50% | 🟢 **LOW (P4)** | No immediate action |

In [ ]:
# --- Priority Classification Function ---
def assign_priority(damage_confidence):
    """Assign rescue priority based on damage prediction confidence."""
    if damage_confidence > 0.90:
        return 'P1 - CRITICAL', '🔴', 'Immediate rescue deployment'
    elif damage_confidence > 0.70:
        return 'P2 - HIGH', '🟠', 'Urgent assessment needed'
    elif damage_confidence > 0.50:
        return 'P3 - MEDIUM', '🟡', 'Schedule inspection'
    else:
        return 'P4 - LOW', '🟢', 'No immediate action'


# Apply priority to test predictions
# Note: class index 0 = 'damage', class index 1 = 'no_damage' (alphabetical)
# sigmoid output > 0.5 => class 1 (no_damage), < 0.5 => class 0 (damage)
# So damage_confidence = 1 - sigmoid_output
damage_confidences = 1 - y_pred_probs.flatten()

# Priority distribution
priority_counts = {'P1 - CRITICAL': 0, 'P2 - HIGH': 0, 'P3 - MEDIUM': 0, 'P4 - LOW': 0}
for conf in damage_confidences:
    priority, _, _ = assign_priority(conf)
    priority_counts[priority] += 1

print("Rescue Priority Distribution (Test Set):")
print("=" * 45)
for priority, count in priority_counts.items():
    pct = count / len(damage_confidences) * 100
    print(f"  {priority}: {count} buildings ({pct:.1f}%)")

In [ ]:
# --- Priority Distribution Bar Chart ---
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71']
priorities = list(priority_counts.keys())
counts = list(priority_counts.values())

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(priorities, counts, color=colors, edgecolor='black', linewidth=0.5)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Rescue Priority Distribution', fontsize=16, fontweight='bold')
ax.set_xlabel('Priority Level', fontsize=12)
ax.set_ylabel('Number of Buildings', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('priority_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: priority_distribution.png")

---
## 9. Sample Predictions with Priority

In [ ]:
# --- Visualize predictions with priority labels ---
test_generator.reset()
batch_images, batch_labels = next(test_generator)

# Get predictions for this batch
batch_preds = model.predict(batch_images, verbose=0)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Predictions with Rescue Priority', fontsize=18, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    if i >= len(batch_images):
        break
    
    # Undo preprocessing for display
    img = batch_images[i].copy()
    img = (img - img.min()) / (img.max() - img.min())  # Normalize to [0, 1]
    
    pred_prob = batch_preds[i][0]
    pred_class = 'no_damage' if pred_prob > 0.5 else 'damage'
    true_class = class_labels[int(batch_labels[i])]
    
    damage_conf = 1 - pred_prob
    priority, emoji, action = assign_priority(damage_conf)
    
    ax.imshow(img)
    
    # Color title based on correctness
    color = 'green' if pred_class == true_class else 'red'
    ax.set_title(f"True: {true_class}\nPred: {pred_class}\n{priority}",
                 fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sample_predictions.png")

---
## 10. Save the Model

In [ ]:
# --- Save the trained model ---
model.save('rescuevision_model.keras')
print("Model saved as: rescuevision_model.keras")

# --- Summary ---
print("\n" + "=" * 60)
print("  RESCUEVISION — PROJECT SUMMARY")
print("=" * 60)
print(f"  Model:           ResNet50 + Transfer Learning")
print(f"  Task:            Binary Classification (damage / no_damage)")
print(f"  Test Accuracy:   {test_accuracy*100:.2f}%")
print(f"  ROC AUC:         {roc_auc:.4f}")
print(f"  Priority Levels: P1 (Critical) → P4 (Low)")
print(f"  Saved Model:     rescuevision_model.keras")
print("=" * 60)

---
## 11. Inference — Predict on a New Image

Use this function to classify any new satellite image.

In [ ]:
def predict_building(image_path, model):
    """
    Predict damage status and rescue priority for a single building image.
    
    Args:
        image_path: Path to the satellite image
        model: Trained Keras model
    
    Returns:
        Dictionary with prediction details
    """
    # Load and preprocess
    img = keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_array = keras.preprocessing.image.img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    pred_prob = model.predict(img_array, verbose=0)[0][0]
    damage_conf = 1 - pred_prob
    pred_class = 'damage' if damage_conf > 0.5 else 'no_damage'
    priority, emoji, action = assign_priority(damage_conf)
    
    # Display
    plt.figure(figsize=(6, 6))
    plt.imshow(Image.open(image_path))
    plt.title(f"Prediction: {pred_class.upper()}\n"
              f"Damage Confidence: {damage_conf*100:.1f}%\n"
              f"{emoji} {priority} — {action}",
              fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    return {
        'image': image_path,
        'prediction': pred_class,
        'damage_confidence': f"{damage_conf*100:.1f}%",
        'priority': priority,
        'action': action
    }

# --- Test on a sample image ---
sample_damage = os.path.join(SPLIT_DIR, 'test', 'damage')
sample_img = os.path.join(sample_damage, os.listdir(sample_damage)[0])
result = predict_building(sample_img, model)
print("\nPrediction Result:")
for k, v in result.items():
    print(f"  {k}: {v}")